# Taller 2 — Reformulación del modelo de programación de lotes de tomate
Este cuaderno reproduce de principio a fin la tabla del informe: genera las instancias E1–E4 con semilla fija, resuelve las formulaciones A y B con HiGHS y reporta los siete indicadores.

**Tiempo estimado de ejecución completa:** puede superar una hora (límite de 600 s × 3 repeticiones × 8 combinaciones en el peor caso). Para una prueba rápida, cambiar `LIMITE_TIEMPO` y `REPETICIONES` en la celda siguiente.

In [ ]:
import os
os.environ["LIMITE_TIEMPO"] = "600"   # segundos por corrida
os.environ["REPETICIONES"] = "3"

import pandas as pd
from datos import generar, TAMANOS, SEMILLA
import experimento as ex
print("Semilla:", SEMILLA)
ex.entorno()

## 1. Instancias

In [ ]:
filas = []
for nombre, n in TAMANOS.items():
    d = generar(n)
    brutas = sum(d["G"].values())
    comerciales = sum(d["Q"].values())
    filas.append({"escenario": nombre, "lotes": n, "gamelas_brutas": brutas,
                  "gamelas_comerciales_esperadas": round(comerciales),
                  "oferta_capacidad_pct": round(100 * brutas / 9600, 1)})
pd.DataFrame(filas)

## 2. Experimento

In [ ]:
resultados = []
for nombre, n in TAMANOS.items():
    print("Resolviendo", nombre)
    resultados += ex.correr_instancia(nombre, n)
df = pd.DataFrame(resultados)
df.to_csv("resultados.csv", index=False)

## 3. Verificación numérica de equivalencia
Los óptimos enteros de A y B deben coincidir en cada instancia resuelta a optimalidad. Si una formulación se detuvo por límite de tiempo, se verifica que sus intervalos [incumbente, cota] se intersecten.

In [ ]:
for nombre in TAMANOS:
    a = df[(df.instancia == nombre) & (df.formulacion == "A")].iloc[0]
    b = df[(df.instancia == nombre) & (df.formulacion == "B")].iloc[0]
    if a.estado == "Optimal" and b.estado == "Optimal":
        ok = abs(a.Z_IP - b.Z_IP) <= 1e-4 * abs(a.Z_IP)
        print(nombre, "óptimos iguales:", ok, a.Z_IP, b.Z_IP)
        assert ok, "Los óptimos enteros difieren: la comparación no es válida"
    else:
        ok = max(a.Z_IP, b.Z_IP) <= min(a.cota_final, b.cota_final) + 1e-6
        print(nombre, "intervalos compatibles (no certificado por límite de tiempo):", ok)

## 4. Tabla del informe

In [ ]:
tabla = df[["instancia", "oferta_pct", "formulacion", "variables", "restricciones", "no_nulos",
            "Z_LP", "Z_IP", "brecha_LP_pct", "integrality_gap", "brecha_opt_pct",
            "nodos", "cortes_generados", "cortes_activos", "tiempo_medio_s", "estado"]].copy()
tabla.round({"Z_LP": 0, "Z_IP": 0, "brecha_LP_pct": 2, "integrality_gap": 4,
             "brecha_opt_pct": 4, "tiempo_medio_s": 2})